In [ ]:
import json
import os
import sys
from pathlib import Path

import torch
import yaml
from IPython.display import Markdown, display

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Gốc dữ liệu: biến môi trường ưu tiên hơn config/url.yaml (url.yaml trỏ tới D:\content trên máy local).
# Chỉ đặt khi thư mục có thật, nên trên máy local không đổi gì.
for _env, _base, _marker in (
    ("MLNC_MIXTURE_36L", "/marimo/datasets/mix-dataset", "labels.txt"),
    ("MLNC_AUDIOSET_40L", "/marimo/datasets/audioset", "metadata/manifest.csv"),
):
    # Kaggle giải nén lồng một cấp: gốc thật là thư mục chứa tệp đánh dấu (giống find_root trong scripts/download_dataset.py)
    _tail = Path(_marker).parts
    _hits = sorted(Path(_base).rglob(_tail[-1]), key=lambda p: len(p.parts)) if Path(_base).is_dir() else []
    _hits = [p for p in _hits if p.parts[-len(_tail) :] == _tail]
    if _hits:
        os.environ.setdefault(_env, str(_hits[0].parents[len(_tail) - 1]))

RAW_CONFIG = yaml.safe_load((ROOT / "config" / "frcrn.yaml").read_text(encoding="utf-8"))

# frcrn-verify: fine-tune FRCRN trên mix-dataset với target noise

Notebook này **mô tả, kiểm tra và (khi bạn bật cờ) chạy** việc fine-tune FRCRN trên mix-dataset, với `target = noise`.
Mọi thứ (giải thích, sơ đồ, cấu hình, ô chạy, ô đọc kết quả) nằm trong chính file này.

**Bài toán.** Dùng mask loại tiếng nói khỏi mixture để lấy nhiễu. Đầu vào mạng là `mixture`, tham chiếu của loss là `noise`.

**Mặc định không chạy gì nặng.** Mọi cờ `RUN_*` đều là `False`. *Run all* chỉ dựng cấu hình và hiển thị.

Nhãn độ chắc chắn: **[đọc]** lấy từ mã hoặc tệp trong repo, **[suy luận]** suy ra từ mã, **[chưa kiểm chứng]** chưa có bằng chứng.

| Mục | Nội dung | Cờ |
| --- | --- | --- |
| 1 | Pipeline fine-tune | |
| 2 | Dữ liệu: mix-dataset | `RUN_DATA_CHECK` |
| 3 | Kiến trúc FRCRN | |
| 4 | `target = noise`: mask lý tưởng | |
| 5 | Hàm loss | `RUN_CHECKS` |
| 6 | Cấu hình fine-tune | |
| 7 | Chạy fine-tune | `RUN_FINETUNE` |
| 8 | Đọc kết quả | |
| 9 | Kiểm duyệt pipeline run | |
| 10 | Checklist trước khi chạy trên molab | |
| 11 | Đã và chưa kiểm chứng | |
| 12 | Chạy toàn bộ pipeline | `RUN_TRAINING` |

## 1. Pipeline fine-tune

Khởi tạo từ checkpoint pretrain FRCRN_SE_16K, huấn luyện trên mix-dataset (mixture đã trộn sẵn), tham chiếu là nhiễu.
Chỉ số chọn best là `noise_si_snr_hi` (trung bình SI-SNR của nhiễu ở 15 và 20 dB, trên validation). **[đọc]** `trial_train_config` trong `scripts/sweep_frcrn.py`.

Pipeline fine-tune FRCRN, target noise, trên mix-dataset

## 2. Dữ liệu: mix-dataset

Dùng lớp `Mixture36Dataset` (`scripts/load_dataset.py`) qua `finetune_frcrn` (`scripts/finetune.py`). **[đọc]**

| Thành phần | Nguồn | Vai trò |
| --- | --- | --- |
| `waveform` | cột `mixture_path` | đầu vào của mạng |
| `noise` | cột `noise_path` (`load_noise=True`) | tham chiếu của loss khi `target = noise`. **Đã được nhân theo SNR** khi tạo mix-dataset (theo bạn), nên là thành phần nhiễu thật nằm trong mixture |
| `clean` | cột `clean_path` (`load_clean=True`) | không dùng ở `target = noise` |
| `snr_db` | cột `target_snr_db` | thống kê `noise_si_snr` theo từng mức SNR, không đi vào loss |

- Split theo doc: train 30.240, validation 6.480, test 6.480 mẫu; SNR −5, 0, 5, 10, 15, 20 dB. Giai đoạn fine-tune chỉ dùng train và validation, **không đụng test**. **[đọc]**
- Cắt đoạn: train 2 s, validation 4 s (`segment_seconds`, `eval_segment_seconds`). **[đọc]**
- Khác với `DynamicMixDataset` (trộn động tiếng nói với nhiễu AudioSet): ở đây mixture là cố định, 6 mức SNR cố định, nên phân phối khớp chính xác validation và test nhưng ít đa dạng hơn. **[suy luận]**
- Với `target = speech`, trainer tính tiếng nói là `mixture − noise`. Với `target = noise` thì **không cần** phép trừ này: tham chiếu là chính `noise_path`. **[đọc]** `training_target` trong `separator.py`.
- Công thức nhân SNR khi tạo mix-dataset **không nằm trong repo** (bộ nạp chỉ đọc file). Nếu cùng công thức với `mix_at_snr` của `DynamicMixDataset` (co nhiễu để công suất nhiễu thấp hơn công suất tiếng nói đúng `snr_db` dB, rồi chia cả mixture và nhiễu cho đỉnh biên độ nếu đỉnh vượt 1) thì `mixture = clean + noise` đúng tới hệ số đỉnh chung. **[chưa kiểm chứng]**; kiểm bằng ô `RUN_DATA_CHECK` bên dưới (so SNR đo từ `clean`/`noise` với nhãn `target_snr_db`).

Luồng dữ liệu từ mix-dataset đến batch

### Thử khôi phục nhiễu tham chiếu từ AudioSet (hướng A)

Yêu cầu: tham chiếu nhiễu lấy từ clip AudioSet gốc, không lấy từ `noise_path`. Đã thử khôi phục `noise_path` từ AudioSet trên 15 hàng validation
(11 hàng thường và 4 hàng `noise_scale` 125–461): clip gốc 44,1 kHz int16, đổi mono, resample 16 kHz, tìm đoạn 4 s khớp nhất bằng tương quan chéo. **[đo]**

| Kết quả | Chi tiết |
| --- | --- |
| Hệ số nhân | hệ số khớp ≈ `noise_scale × post_gain`, lệch ≤ ~1% ở cả 15 hàng (kể cả hàng cực đoan) |
| Dạng sóng | sai số năng lượng tương đối: tốt nhất 8e-6 đến 2e-5 (2 hàng); điển hình 7e-4 đến 2e-2 (7 hàng); kém 5% đến 24% (6 hàng, gồm cả 4 hàng cực đoan) |
| Kênh và resample | trung bình hai kênh tốt hơn kênh trái hoặc phải; `resample_poly`, FFT, torchaudio, soxr cho sai số gần như nhau |
| Đoạn cắt | không có quy tắc đơn giản (0,0 s, 6,0 s và các giá trị ngẫu nhiên); cắt ở miền 44,1 kHz rồi resample không cải thiện; `random_seed` của hàng không tái lập được offset bằng 4 cách rút thông thường (`default_rng.integers`, `RandomState.randint`, `random.Random.randint`, `default_rng.random × slack`) trên 8 hàng |

Kết luận: **chưa khôi phục chính xác được** nhiễu từ AudioSet; script sinh mix-dataset không có trong repo. Sai số tương đối ε giới hạn SI-SNR đo được ở khoảng `−10·log10(ε)` dB
(ε = 5% cho ≈ 13 dB, ε = 24% cho ≈ 6 dB). Vì vậy các ô bên dưới **vẫn đọc nhiễu từ `noise_path`**; hướng khôi phục từ AudioSet chưa được đưa vào ô chạy. **[suy luận từ công thức]**

In [ ]:
RUN_DATA_CHECK = False  # True: đọc ~12 mẫu validation từ mix-dataset (cần dataset trên máy), kiểm mixture = clean + noise

if not RUN_DATA_CHECK:
    print("RUN_DATA_CHECK = False: bỏ qua. Đặt True để kiểm tra dữ liệu.")
else:
    from scripts.load_dataset import Mixture36Dataset

    try:
        dataset = Mixture36Dataset(split="validation", duration=2.0, load_noise=True, load_clean=True)
    except Exception as error:
        print("Không mở được mix-dataset:", repr(error))
    else:
        print(f"validation: {len(dataset)} mẫu")
        print("chỉ mục | SNR nhãn | SNR đo từ clean/noise | ||mixture-clean-noise||² / ||mixture||²")
        for index in range(0, len(dataset), max(len(dataset) // 12, 1)):
            item = dataset[index]
            mixture, noise, clean = item["waveform"], item["noise"], item["clean"]
            residual = (mixture - clean - noise).square().sum() / mixture.square().sum()
            measured = 10 * torch.log10(clean.square().sum() / noise.square().sum())
            print(f"{index:7d} | {float(item['snr_db']):8.1f} | {float(measured):21.2f} | {float(residual):.2e}")

## 3. Kiến trúc FRCRN

Lớp bọc `FRCRNSeparator` chuẩn hoá biên độ rồi gọi mạng `FRCRN` (`src/model/frcrn/`). **[đọc]**

1. `audio_norm`: đưa mixture về mức −25 dB, giữ lại hệ số `scale`.
2. `ConvSTFT`: cửa sổ 640, hop 320, 321 bin; tách thực và ảo.
3. `UNet` 1 nhận `[B, 1, 321, T, 2]` rồi trả mask thô; `UNet` 2 nhận đầu ra của `UNet` 1.
4. **Mask cuối = tanh(U1) + tanh(U2(U1))**, giá trị trong (−2, 2).
5. Nhân phức mask với phổ vào, `ConviSTFT` + overlap-add, cắt về độ dài gốc, nhân lại `scale`.
6. Với `target = noise`, `forward` trả **thẳng** đầu ra này (n̂), không trừ khỏi mixture.

Mạng giống hệt khi `target = speech`; chỉ khác ý nghĩa của đầu ra và mục tiêu huấn luyện.

Luồng xuôi của FRCRN với target noise

Cấu trúc một U-Net

## 4. `target = noise`: mask lý tưởng

Với phổ phức của mixture `Y = S + N`:

| | `target = speech` | `target = noise` |
| --- | --- | --- |
| Mạng học | `M̂·Y ≈ S` | `M̂·Y ≈ N` |
| Mask lý tưởng | `S / Y` | `N / Y = 1 − S/Y` |
| `estimate` của mạng | tiếng nói ŝ | nhiễu n̂ |
| `forward` trả về | `mixture − ŝ` | `n̂` |
| Tham chiếu của loss | `mixture − noise` (đã chuẩn hoá) | `noise / scale` |

Ví dụ một ô: `S = 0,6+0,8j`, `N = 0,2−0,1j` nên `Y = 0,8+0,7j`. Khi đó mask tiếng nói `= 0,920+0,195j`, mask nhiễu `= 0,080−0,195j`, tổng bằng 1.

Ở SNR cao nhiễu yếu nên mask nhiễu nhỏ: ví dụ `|S| ≫ |N|` cho mask nhiễu ≈ `0,039−0,018j`; ở SNR thấp (`|S| ≪ |N|`) mask nhiễu ≈ `0,529−0,118j`. **[tính tay]**

Lưu ý khi chuyển từ `speech` sang `noise` **[suy luận]**:
- Checkpoint pretrain được huấn luyện cho tách tiếng nói, nên mask ban đầu gần `S/Y`, còn mục tiêu mới là `1 − S/Y`: mạng phải học lại. Điểm epoch 0 (baseline) sẽ thấp.
- Bằng chứng sweep về `noise` còn yếu: hai lượt chỉ chạy 2 epoch rồi bị dừng sớm (−3,67 và −6,21 dB), khác cấu hình nhau, nên chưa kết luận được. **[đọc, theo doc]**

## 5. Hàm loss

`FRCRNLoss` (`src/loss/frcrn.py`). **[đọc]** Gọi `normalized` là mixture đã chuẩn hoá, `n` là nhiễu chuẩn hoá `noise / scale`, `n̂` là đầu ra của mạng:

$$\mathcal{L}= w_m\cdot 640\Big[\mathrm{MSE}(\hat M_{re},M^*_{re})+\mathrm{MSE}(\hat M_{im},M^*_{im})\Big]\;-\;\overline{\mathrm{SI\text{-}SNR}(n,\hat n)},\qquad M^*=\frac{N\,\overline{Y}}{|Y|^2+\varepsilon}$$

- Mask lý tưởng cắt giá trị ngoài ±2 (`mask_bound`) về ±1. `mask_weight` $w_m$ mặc định 1,0.
- MSE là trung bình trên toàn bộ `B × 321 × T` phần tử, tính riêng cho nửa thực và nửa ảo rồi cộng.
- **SI-SNR**: trừ trung bình, chiếu `n̂` lên `n` ($\alpha = \langle \hat n, n\rangle / \lVert n\rVert^2$), phần dư $e=\hat n-\alpha n$, rồi $10\log_{10}\big(\lVert \alpha n\rVert^2 / \lVert e\rVert^2\big)$, thêm `eps = 1e-6`.
- Bất biến với hệ số khuếch đại, đảo dấu và lệch DC. Không bị chặn trên (tối đa ≈ 96 dB ở `eps = 1e-6` với đoạn 4.000 mẫu). **[chạy thử]**
- Thang của hai thành phần: nếu mỗi MSE bằng 0,01 thì mask loss ≈ 12,8 (với `mask_weight = 1`), có thể lớn hơn nhiều so với SI-SNR (cỡ chục dB). Trong sweep, `loss` trên W&B ≈ 570–616 khi `mask_weight = 4`, tức mask loss chiếm gần hết giá trị. **[suy luận từ số liệu W&B]**
- Loss không phụ thuộc `snr_db`; nhãn SNR chỉ dùng để thống kê lúc đánh giá.

Luồng tính loss với target noise

In [ ]:
RUN_CHECKS = False  # True: kiểm tra số học trên CPU với dữ liệu ngẫu nhiên (nhẹ, không đọc dữ liệu thật, không ghi tệp)

if not RUN_CHECKS:
    print("RUN_CHECKS = False: bỏ qua. Đặt True để chạy kiểm tra.")
else:
    from src.loss.frcrn import FRCRNLoss, complex_ideal_ratio_mask, si_snr
    from src.model.frcrn import FRCRNSeparator
    from src.model.frcrn.config import FRCRNConfig

    config = FRCRNConfig.from_dict({**RAW_CONFIG[FRCRNConfig.section], "target": "noise"})
    separator = FRCRNSeparator(config, FRCRNLoss(config, **RAW_CONFIG.get("loss", {}))).eval()

    torch.manual_seed(0)
    speech, noise = torch.randn(2, 16000) * 0.1, torch.randn(2, 16000) * 0.03
    mixture = speech + noise

    print("tham số học được:", f"{sum(p.numel() for p in separator.network.parameters()):,}")
    print("training_target(noise) == noise:", bool(torch.equal(separator.training_target(mixture, noise), noise)))
    with torch.no_grad():
        estimate = separator(mixture)
        loss_value = separator.loss(mixture, noise).item()
    print("forward:", tuple(estimate.shape), "| loss trên dữ liệu ngẫu nhiên:", round(loss_value, 3), "(chỉ kiểm tra chạy được)")

    bins = config.num_bins
    big = 1e6  # không cắt, để kiểm đẳng thức đại số
    total = complex_ideal_ratio_mask(noise, mixture, config, mask_bound=big) + complex_ideal_ratio_mask(
        speech, mixture, config, mask_bound=big
    )
    print("mask nhiễu + mask tiếng nói: sai lệch thực so với 1 =", float((total[:, :bins] - 1).abs().max()),
          "| sai lệch ảo so với 0 =", float(total[:, bins:].abs().max()))

    estimate_toy = noise + 0.1 * torch.randn_like(noise)
    for name, value in (("est", estimate_toy), ("3 * est", 3 * estimate_toy), ("-est", -estimate_toy)):
        print(f"SI-SNR(noise, {name}) = {si_snr(noise, value).mean().item():.3f} dB")

## 6. Cấu hình fine-tune

Cấu hình đọc từ `config/frcrn.yaml` của máy này, rồi áp `SETTINGS` bằng `trial_train_config` (chọn best theo `noise_si_snr_hi`). **[đọc]**

Hai lựa chọn nền (`BASE_CHOICE`):
- `"yaml"`: giữ giá trị trong `config/frcrn.yaml` (mặc định của repo), chỉ đổi `target = noise`.
- `"sweep_best"`: bộ tốt nhất của sweep W&B (lượt `xqrcbdzg`). Bộ này được dò cho `target = speech`, nên **thiên lợi cho `speech`**; dùng cho `noise` là điểm xuất phát chứ chưa phải tối ưu. **[đọc, theo doc]**

Cấu hình trên molab khác bản local (đã thấy: molab `epochs 100, batch 12`). Hãy đối chiếu trước khi chạy.

In [ ]:
from scripts.sweep_frcrn import ORIGINAL_CHECKPOINT, best_epoch_scores, run_trial, trial_train_config
from src.train import FRCRNTrainConfig

BASE_CHOICE = "yaml"  # "yaml" hoặc "sweep_best"
SWEEP_BEST = {
    "lr": 0.000273365064091293,
    "lr_decay": 1.0,
    "halve_lr_after": 2,
    "batch_size": 12,
    "segment_seconds": 2.0,
    "grad_clip": 5.0,
    "mask_weight": 4.0,
}
SETTINGS = {"target": "noise", **(SWEEP_BEST if BASE_CHOICE == "sweep_best" else {})}

base_config = FRCRNTrainConfig.from_yaml(ROOT / "config" / "frcrn.yaml")
train_config = trial_train_config(base_config, SETTINGS)
_fields = (
    "epochs", "patience", "lr", "lr_decay", "halve_lr_after", "grad_clip", "batch_size", "eval_batch_size",
    "segment_seconds", "eval_segment_seconds", "snr_levels", "selection_metric", "high_snr_levels",
)
_rows = "\n".join(f"| `{k}` | `{getattr(train_config, k)}` |" for k in _fields)
display(
    Markdown(
        f"""**Nền:** `{BASE_CHOICE}` | **target:** `{SETTINGS['target']}` | **mask_weight:** `{SETTINGS.get('mask_weight', 1.0)}`
| Tham số | Giá trị |
| --- | --- |
{_rows}

Loss (từ yaml): `{RAW_CONFIG.get('loss')}` | checkpoint khởi tạo: `{ORIGINAL_CHECKPOINT}` (tồn tại: {ORIGINAL_CHECKPOINT.is_file()})"""
    )
)

## 7. Chạy fine-tune

`RUN_FINETUNE = False` mặc định, và ô còn bị chặn bởi `ALLOW_NOISE_PATH_REFERENCE = False` (xem mục 9, phát hiện 1). Khi mở, ô dưới gọi `run_trial` (`scripts/sweep_frcrn.py`), tức `finetune_frcrn` trên `Mixture36Dataset` với `target = noise`. **[đọc]**

Trước khi bật:
- Cần có mix-dataset giải nén sẵn và checkpoint pretrain `pretrained/last_best_checkpoint.pt` (FRCRN_SE_16K gốc).
- Thời gian mỗi epoch chưa đo trên toàn bộ dữ liệu (cấu hình sweep ghi "~30 phút/epoch, ước tính thô"); dùng `MAX_SAMPLES = 1200` để chạy thử nhanh như sweep. **[đọc]**
- Ghi vào `output/` (sao lưu Drive nếu `USE_DRIVE = True`). Không ghi đè lần chạy cũ vì `RunPaths.new` tạo thư mục mới theo thời gian.

In [ ]:
RUN_FINETUNE = False  # True: bắt đầu fine-tune (tốn GPU và thời gian)
# Yêu cầu hiện tại: tham chiếu nhiễu lấy từ AudioSet, KHÔNG lấy từ noise_path của mix-dataset.
# Đường chạy bên dưới (finetune_frcrn -> Mixture36Dataset) vẫn đọc noise_path, nên bị chặn mặc định.
# Chỉ đặt True khi bạn chủ động chấp nhận dùng noise_path (trái yêu cầu trên).
ALLOW_NOISE_PATH_REFERENCE = False
SMOKE = True  # True: 1 epoch, 64 mẫu để kiểm đường chạy; False: chạy thật
USE_DRIVE = False  # True: đăng nhập Google Drive, sao lưu best.pt/resume.pt/result.json
DRIVE_DIR = "FA26_Capstone/Truong/frcrn-noise"  # thư mục Drive MỚI (không trùng thư mục có sẵn)
MAX_SAMPLES = None  # None: toàn bộ train/validation; 1200: như sweep
NUM_WORKERS = 4

result = None
if not RUN_FINETUNE:
    print("RUN_FINETUNE = False: bỏ qua. Không chạy gì.")
elif not ALLOW_NOISE_PATH_REFERENCE:
    print("Không chạy: tham chiếu nhiễu hiện vẫn đọc từ noise_path (mix-dataset), trái yêu cầu lấy từ AudioSet. Xem mục 9.")
elif not ORIGINAL_CHECKPOINT.is_file():
    print(f"Thiếu checkpoint pretrain: {ORIGINAL_CHECKPOINT}")
else:
    from src.utils.logger import setup_logger
    from src.utils.paths import RunPaths

    run = RunPaths.new("frcrn-noise", ROOT / "output").create()
    logger = setup_logger(run.run_id, run.log_file)
    logger.info("run %s | target=%s | base=%s | max_samples=%s", run.run_id, SETTINGS["target"], BASE_CHOICE, MAX_SAMPLES)

    fs = None
    if USE_DRIVE:
        from gdrive_fsspec import GoogleDriveFileSystem

        fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False})

    result = run_trial(
        SETTINGS,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        trial_dir=run.checkpoint_dir,
        checkpoint=ORIGINAL_CHECKPOINT,
        base_config=base_config,
        max_samples=MAX_SAMPLES,
        num_workers=NUM_WORKERS,
        smoke=SMOKE,
        fs=fs,
        drive_dir=DRIVE_DIR,
    )
    run.history_path("frcrn_noise").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("best_epoch:", result["best_epoch"], "| best_score:", result["best_score"], "| lưu tại", run.history_path("frcrn_noise"))

## 8. Đọc kết quả

Chỉ số chính: `noise_si_snr` của nhiễu thu được so với nhiễu gốc, **tách theo từng mức SNR** và `noise_si_snr_hi` (15 và 20 dB).

Mốc tham chiếu trên **test** (6.480 mẫu), SI-SNR nhiễu, dB, theo `doc/frcrn-finetune-history.md` mục 6: **[đọc]**

| Checkpoint | −5 | 0 | 5 | 10 | 15 | 20 | 15–20 | Trung bình |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| FRCRN_SE_16K gốc | 9,14 | 7,00 | 5,37 | 2,99 | −1,48 | −4,46 | −2,97 | 3,09 |
| `best.pt` cũ | 13,79 | 11,06 | 8,71 | 6,43 | 3,31 | 0,60 | +1,95 | 7,32 |
| fine-tune thẳng trên Mixture-36l (epoch 9) | 9,72 | 8,14 | 6,73 | 4,83 | 1,32 | −1,46 | −0,07 | 4,88 |

Cách đọc thang:
- 0 dB: năng lượng phần sai bằng phần đúng; 3 dB: sai bằng một nửa; 10 dB: sai 10%; 20 dB: sai 1%.
- "Không lọc" (coi nguyên mixture là nhiễu) cho SI-SNR ≈ **−SNR của mixture** nếu tiếng nói và nhiễu không tương quan: +5, 0, −5, −10, −15, −20 dB. Từ đó tính SI-SNRi. **[suy luận, chưa đo]**
- Mỗi con số chỉ 1 seed. Phải đo độ lệch chuẩn giữa các seed trước khi kết luận cách nào hơn.
- Thước đo cuối cùng của bài toán là phân loại nhiễu (macro-F1, mAP của BEATs), không phải SI-SNR.

In [ ]:
RESULT_JSON = None  # đường dẫn tới history frcrn_noise.json hoặc result.json sau khi chạy xong

_data = result
if _data is None and RESULT_JSON:
    _data = json.loads(Path(RESULT_JSON).read_text(encoding="utf-8"))

if _data is None:
    print("Chưa có kết quả: chạy mục 7 hoặc đặt RESULT_JSON.")
else:
    scores = best_epoch_scores(_data)
    levels = (-5, 0, 5, 10, 15, 20)

    def _score(level):
        for key in (f"noise_si_snr_{level}", f"noise_si_snr_{float(level)}"):
            if key in scores:
                return scores[key]
        return float("nan")

    header = "| | " + " | ".join(str(v) for v in levels) + " | 15–20 | trung bình |\n| --- |" + " --- |" * (len(levels) + 2)
    measured = " | ".join(f"{_score(v):.2f}" for v in levels)
    # "không lọc" (coi mixture là nhiễu): SI-SNR ≈ -SNR; hi = trung bình 15 và 20 dB, cuối = trung bình 6 mức
    unfiltered = " | ".join(f"{-v:+d}" for v in levels)
    unfiltered_hi = -sum(v for v in levels if v in (15, 20)) / 2
    unfiltered_mean = -sum(levels) / len(levels)
    display(
        Markdown(
            f"best epoch **{_data['best_epoch']}** / {_data['epochs_run']} | chọn theo `{_data['selection_metric']}` = {_data['best_score']:.3f}\n\n"
            f"{header}\n| noise_si_snr (validation) | {measured} | {scores.get('noise_si_snr_hi', float('nan')):.2f} | {scores.get('noise_si_snr', float('nan')):.2f} |\n"
            f"| không lọc (ước tính: −SNR) | {unfiltered} | {unfiltered_hi:.1f} | {unfiltered_mean:.1f} |"
        )
    )

## 9. Kiểm duyệt pipeline run

Rà lại đường chạy `run_trial` → `finetune_frcrn` → `FRCRNTrainer` trước khi chạy thật. Mức: **chặn** (không chạy được hoặc sai yêu cầu), **cảnh báo**, **ghi chú**.

| # | Phát hiện | Mức | Việc cần làm |
| --- | --- | --- | --- |
| 1 | Tham chiếu nhiễu vẫn đọc từ `noise_path` của mix-dataset, **trái yêu cầu** lấy nhiễu từ AudioSet **[đọc]** | chặn | Ô chạy bị chặn bởi `ALLOW_NOISE_PATH_REFERENCE = False`. Cần tham chiếu từ AudioSet, hoặc bạn chủ động đặt `True` |
| 2 | Chưa khôi phục chính xác nhiễu từ AudioSet: sai số 8e-6 đến 24% trên 15 hàng; script sinh dữ liệu không có trong repo; `random_seed` không tái lập được đoạn cắt (thử 4 cách rút thông thường trên 8 hàng) **[đo]** | chặn | Chọn: lọc hàng theo sai số, hoặc dùng `DynamicMixDataset` (nhiễu từ AudioSet, chính xác theo cấu tạo) |
| 3 | Sandbox molab **thiếu `soundfile`, `torchaudio`, `timm`**: `scripts.load_dataset`, `src.train` không import được **[đo trên sandbox]** | chặn | Cài trong môi trường rồi khởi động lại kernel. Không cài giữa phiên: cài `timm` từng nâng `torch` dưới kernel đang chạy **[đọc doc]** |
| 4 | `config/url.yaml` trỏ `D:\content\...`; trên molab chưa có dataset, checkpoint pretrain **[đọc]** | chặn | Đặt `MLNC_MIXTURE_36L`, `MLNC_AUDIOSET_40L` (ô setup tự đặt khi thư mục `/marimo/datasets/...` có thật); tải `pretrained/last_best_checkpoint.pt` |
| 5 | Train luôn cắt **2 s đầu** của clip 4 s (`read_audio` với `rng=None` cho `start = 0`); validation dùng đủ 4 s. `mixture` và `noise` cắt cùng vị trí nên không lệch pha **[đọc]** | ghi chú | Nửa sau mỗi clip không bao giờ được dùng khi train. Tham chiếu từ AudioSet (nếu có) phải dùng cùng offset |
| 6 | `USE_DRIVE = False` mà sandbox mất khi hết phiên: không có `resume.pt` **[đọc doc]** | cảnh báo | Bật Drive với thư mục mới; chạy đầu tiên dùng `SMOKE = True` (1 epoch, 64 mẫu) |
| 7 | `src/utils/download.py` chứa token Kaggle viết thẳng, đã commit **[đọc]** | cảnh báo | Tạo lại token; file này **không** được đẩy lên molab |

Trạng thái từng khối của pipeline run

## 10. Checklist trước khi chạy trên molab

Sandbox molab là môi trường tạm: hết phiên (giới hạn 12 giờ) hoặc bị xoá thì mất toàn bộ `output/`, trừ những gì đã đẩy ra ngoài. **[đọc, theo doc và log của repo]**

1. Đưa **mã local** lên sandbox: mã `src/` trên molab từng là bản cũ (không có `selection_metric` và `resume_path`) và thiếu `scripts/drive_sync.py`. **[đọc]**
2. Bật `USE_DRIVE = True` và dùng thư mục Drive **mới** (`check_drive_dir` từ chối các thư mục đã có), để có `resume.pt` và `best.pt`.
3. Đặt token (Kaggle, W&B) qua biến môi trường, **không** ghi vào ô code.
4. Tải `result.json`, lịch sử epoch và `best.pt` về máy ngay sau khi chạy xong.
5. Ghi lại `run_id`, commit (`git rev-parse HEAD`), seed và cấu hình dùng (`SETTINGS`, `BASE_CHOICE`) cùng với kết quả.
6. Đo 50 bước huấn luyện thật trước khi ước lượng thời gian toàn bộ.

## 11. Đã và chưa kiểm chứng

**Đã kiểm chứng từ mã và tệp trong repo [đọc]**
- Mạng, mask `tanh + tanh`, loss `mask loss − SI-SNR`, `training_target = noise / scale` khi `target = noise`, `forward` trả thẳng nhiễu.
- `finetune_frcrn` dùng `Mixture36Dataset(load_noise=True)` nên đúng với "mixture vào, nhiễu làm tham chiếu".
- Chỉ số chọn best (`noise_si_snr_hi`) và cách tách theo SNR.

**Chưa kiểm chứng**
- Hiệu quả của `target = noise` so với `speech` trên dữ liệu này: chưa có lần chạy đủ epoch, đủ seed nào.
- Thời gian mỗi epoch trên toàn bộ dữ liệu (chỉ có ước tính thô).
- Công thức nhân SNR của `noise_path` (theo bạn đã áp dụng khi tạo mix-dataset) và việc `mixture = clean + noise` đúng bằng số: không có trong repo. Với `target = noise` điều này quyết định tham chiếu có đúng thang với mixture hay không, nên nên kiểm bằng `RUN_DATA_CHECK` (SNR đo so với nhãn, phần dư `mixture − clean − noise`) trước khi chạy thật.
- Đẳng thức mask nhiễu + mask tiếng nói = 1 bằng số: kiểm bằng `RUN_CHECKS` (mới được chứng minh đại số).
- Nguồn gốc của `best.pt` cũ (+1,95 dB ở 15–20 dB): chưa xác nhận do cấu hình nào tạo ra.

## 12. Chạy toàn bộ pipeline

Ô cuối: `RUN_TRAINING = False` mặc định. Đặt `True` để gọi `run_all()`: tải dữ liệu (cần biến môi trường `KAGGLE_ACCESS_TOKEN` nếu chưa có dữ liệu),
fine-tune `target: noise` với nhiễu tham chiếu dựng từ AudioSet, **hiện tiến trình từng epoch** (thanh tqdm, `train_loss`, `val_loss`,
`noise_si_snr`, `noise_si_snr_hi`, `lr`, thời gian) và sao lưu `best.pt`/`resume.pt` lên Drive. Chạy trong chính kernel nên kernel bận tới khi xong. Mỗi lần có kết quả tốt nhất mới, ngoài `best.pt` (bản mới nhất) còn lưu `best_e<epoch>_<UTC>.pt` lên Drive; hồ sơ `run_manifest.json` (mốc thời gian từng epoch, vị trí Drive, mã băm code đã dùng) được ghi local và lên Drive. Muốn tái triển khai khi sandbox mất kết nối: `python -m scripts.molab_deploy record|push|launch|status` (xem docstring của `scripts/molab_deploy.py`).

In [ ]:
RUN_TRAINING = False  # True: chạy toàn bộ pipeline (tải dữ liệu, fine-tune target noise với nhiễu từ AudioSet, lưu Drive)
TRAIN_SETTINGS = {
    "epochs": 40,  # số epoch tối đa; kết thúc sớm nếu validation không cải thiện (patience trong config/frcrn.yaml)
    "max_samples": None,  # số mẫu mỗi epoch; None: toàn bộ (24.246 train, 5.118 validation sau lọc)
    "max_error": 1e-2,  # giữ hàng có sai số khôi phục nhiễu AudioSet <= 1% năng lượng
    "use_drive": True,  # sao lưu best.pt, resume.pt, result.json lên Drive (cần token Drive đã đăng nhập)
    "drive_dir": "FA26_Capstone/Truong/frcrn-noise-run1",  # thư mục Drive MỚI (frcrn-noise đang chứa kết quả smoke)
    "work_dir": "output/frcrn-noise-run1",  # tương đối so với thư mục gốc repo (/marimo trên sandbox)
    "smoke": False,  # True: 1 epoch, 64 mẫu để kiểm đường chạy
    "workers": 8,
}

# Module repo cần nạp lại theo thứ tự phụ thuộc: kernel giữ module đã import nên mã mới đẩy lên đĩa sẽ không tự có hiệu lực
REPO_MODULES = (
    "scripts.drive_sync",
    "scripts.load_dataset",
    "scripts.download_dataset",
    "scripts.audioset_noise_ref",
    "src.train.frcrn_trainer",
    "src.train",
    "scripts.finetune",
    "scripts.sweep_frcrn",
    "scripts.run_manifest",
    "scripts.run_frcrn_noise",
)


def run_all(settings=TRAIN_SETTINGS):
    """Tải dữ liệu, fine-tune FRCRN (target noise, nhiễu tham chiếu từ AudioSet), hiện tiến trình từng epoch, lưu Drive."""
    import importlib
    import os
    import sys
    import time
    from pathlib import Path

    from tqdm.auto import tqdm

    importlib.invalidate_caches()
    reloaded = [name for name in REPO_MODULES if name in sys.modules]
    for name in reloaded:
        importlib.reload(sys.modules[name])
    print(f"đã nạp lại {len(reloaded)} module repo")
    for env_name, marker in (("MLNC_MIXTURE_36L", "labels.txt"), ("MLNC_AUDIOSET_40L", "metadata/manifest.csv")):
        root = os.environ.get(env_name)
        if root and not (Path(root) / marker).is_file():  # gốc sai (Kaggle giải nén lồng một cấp): để ensure_dataset dò lại
            print(f"bỏ {env_name}={root}: thiếu {marker}")
            os.environ.pop(env_name)
    from scripts.run_frcrn_noise import ROOT, main

    work_dir = Path(settings["work_dir"])
    work_dir = work_dir if work_dir.is_absolute() else ROOT / work_dir
    finished = work_dir / "checkpoint" / "frcrn" / "result.json"
    if finished.is_file() and not settings.get("allow_finished", False):
        raise FileExistsError(f"{finished} đã có: lần chạy coi như xong và sẽ không huấn luyện. Đổi work_dir và drive_dir sang thư mục mới.")

    start = time.perf_counter()
    total = settings["epochs"]
    bar = tqdm(total=total, desc="epoch", unit="epoch")

    def on_epoch(epoch, scores):
        bar.update(1)
        bar.set_postfix(
            train_loss=f"{scores['train_loss']:.1f}", val_loss=f"{scores['loss']:.1f}", si_snr_hi=f"{scores['noise_si_snr_hi']:.2f}"
        )
        tqdm.write(
            f"epoch {epoch}/{total} | train_loss {scores['train_loss']:.2f} | val_loss {scores['loss']:.2f}"
            f" | noise_si_snr {scores['noise_si_snr']:.2f} dB | noise_si_snr_hi {scores['noise_si_snr_hi']:.2f} dB"
            f" | lr {scores['lr']:.1e} | {(time.perf_counter() - start) / 60:.1f} phút"
        )

    argv = ["--workers", str(settings["workers"]), "--max-error", str(settings["max_error"]), "--drive-dir", settings["drive_dir"]]
    argv += ["--work-dir", str(work_dir)]
    if settings["use_drive"]:
        argv.append("--use-drive")
    if settings["smoke"]:
        argv.append("--smoke")
    if settings["epochs"]:
        argv += ["--epochs", str(settings["epochs"])]
    if settings["max_samples"]:
        argv += ["--max-samples", str(settings["max_samples"])]
    try:
        return main(argv, on_epoch=on_epoch)
    finally:
        bar.close()

if RUN_TRAINING:
    run_all()
else:
    print("RUN_TRAINING = False: chưa chạy. Đặt True rồi chạy lại ô này để bắt đầu (xem mục 9 và 12).")